# ✈️ Flight Delay Prediction
## EDA + Feature Engineering + Regression Model
---
**Dataset:** 2015 Flight Delays & Cancellations — Kaggle  
**Objetivo:** Estimar los minutos de retraso de llegada de un vuelo  
**Modelos:** Random Forest Regressor · XGBoost Regressor  

### 📥 Instrucciones para Google Colab + Drive
Este notebook esta preparado para trabajar directamente desde Google Drive.

Estructura esperada en Drive para los CSV:

`MyDrive/fligts/`

Dentro de esa carpeta deben estar:

- `flights.csv`
- `airlines.csv`
- `airports.csv`

El proyecto seguira usando esta carpeta para guardar el modelo empaquetado:

`MyDrive/flight-delay-prediction/models/flight_delay_model.joblib`

Si los CSV no existen, el notebook intentara descargarlos automaticamente desde Kaggle usando `kagglehub` y copiarlos a `MyDrive/fligts/`. Si Kaggle pide autenticacion, descarga manualmente el dataset desde https://www.kaggle.com/datasets/usdot/flight-delays y sube los tres CSV a `MyDrive/fligts/`.


In [ ]:
import subprocess, sys

libs = ['pandas', 'numpy', 'matplotlib', 'seaborn',
        'scikit-learn', 'xgboost', 'plotly',
        'kagglehub', 'joblib']

for lib in libs:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', lib, '-q'])

print('✅ Todas las librerías instaladas')

In [ ]:
from pathlib import Path
import os
import shutil

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/flight-delay-prediction')
    DATA_DIR = Path('/content/drive/MyDrive/fligts')
else:
    PROJECT_ROOT = Path.cwd()
    DATA_DIR = PROJECT_ROOT / 'fligts'

MODEL_DIR = PROJECT_ROOT / 'models'
SRC_DIR = PROJECT_ROOT / 'src'

for directory in [PROJECT_ROOT, DATA_DIR, MODEL_DIR, SRC_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

os.chdir(PROJECT_ROOT)

print(f'Proyecto activo en: {PROJECT_ROOT}')
print(f'Datos esperados en: {DATA_DIR}')
print(f'Modelos en: {MODEL_DIR}')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor

plt.style.use('dark_background')
BLUE = '#1e90ff'
CYAN = '#00d4ff'
RED = '#f85149'
GREEN = '#3fb950'
GRAY = '#8b949e'

print('✅ Imports listos')

## 📂 1. Carga de Datos

In [ ]:
import kagglehub

required_files = ['flights.csv', 'airlines.csv', 'airports.csv']
missing_files = [filename for filename in required_files if not (DATA_DIR / filename).exists()]

if missing_files:
    print('Faltan archivos en Drive:', missing_files)
    print('Intentando descargar dataset publico desde Kaggle con kagglehub...')

    try:
        downloaded_path = Path(kagglehub.dataset_download('usdot/flight-delays'))
        print(f'Dataset descargado en cache: {downloaded_path}')

        for filename in required_files:
            source_file = next(downloaded_path.rglob(filename), None)
            if source_file is None:
                raise FileNotFoundError(f'No se encontro {filename} dentro de la descarga de Kaggle.')
            shutil.copy2(source_file, DATA_DIR / filename)
            print(f'Copiado a Drive: {DATA_DIR / filename}')

    except Exception as exc:
        raise RuntimeError(
            'No se pudo descargar automaticamente el dataset. '
            'Descargalo manualmente desde https://www.kaggle.com/datasets/usdot/flight-delays '
            'y sube flights.csv, airlines.csv y airports.csv a '
            f'{DATA_DIR}'
        ) from exc
else:
    print('✅ Dataset encontrado en Google Drive')

for filename in required_files:
    file_path = DATA_DIR / filename
    print(f'{filename}: {file_path} | existe={file_path.exists()}')

In [ ]:
SAMPLE = True
SAMPLE_N = 500_000

flights_path = DATA_DIR / 'flights.csv'
airlines_path = DATA_DIR / 'airlines.csv'
airports_path = DATA_DIR / 'airports.csv'

print(f'Cargando flights.csv desde: {flights_path}')
if SAMPLE:
    df_raw = pd.read_csv(
        flights_path,
        low_memory=False,
        skiprows=lambda i: i > 0 and np.random.rand() > SAMPLE_N / 5_819_079,
    )
else:
    df_raw = pd.read_csv(flights_path, low_memory=False)

airlines = pd.read_csv(airlines_path)
airports = pd.read_csv(airports_path)

print(f'✅ Vuelos cargados: {len(df_raw):,} registros')
print(f'   Columnas: {df_raw.shape[1]}')
print(f'   Aerolíneas: {df_raw.AIRLINE.nunique()}')
print(f'   Aeropuertos origen: {df_raw.ORIGIN_AIRPORT.nunique()}')
df_raw.head(3)

## 🔍 2. Análisis Exploratorio de Datos (EDA)

In [ ]:
print('=== TIPOS DE DATOS ==='  )
print(df_raw.dtypes)
print()
print('=== NULOS POR COLUMNA ===')
nulls = df_raw.isnull().sum()
nulls_pct = (nulls / len(df_raw) * 100).round(2)
null_df = pd.DataFrame({'Nulos': nulls, '% Nulo': nulls_pct})
print(null_df[null_df.Nulos > 0].sort_values('% Nulo', ascending=False))
print()
print('=== ESTADÍSTICAS DESCRIPTIVAS: ARRIVAL_DELAY ==='  )
print(df_raw['ARRIVAL_DELAY'].describe())

In [ ]:
df_plot = df_raw[df_raw['ARRIVAL_DELAY'].notna()].copy()

delayed_pct = (df_plot['ARRIVAL_DELAY'] > 15).mean() * 100
on_time_pct = 100 - delayed_pct
cancelled_pct = df_raw['CANCELLED'].mean() * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Distribución de Retrasos — Dataset 2015', color='white', fontsize=14, fontweight='bold')

ax1 = axes[0]
sizes  = [on_time_pct, delayed_pct, cancelled_pct]
labels = [f'A tiempo\n{on_time_pct:.1f}%',
          f'Retrasado >15min\n{delayed_pct:.1f}%',
          f'Cancelado\n{cancelled_pct:.1f}%']
colors = [GREEN, RED, GRAY]
ax1.pie(sizes, labels=labels, colors=colors, startangle=90,
        textprops={'color': 'white', 'fontsize': 10})
ax1.set_title('% Vuelos por estado', color='white')

ax2 = axes[1]
delayed_only = df_plot[df_plot['ARRIVAL_DELAY'] > 15]['ARRIVAL_DELAY']
ax2.hist(delayed_only.clip(upper=200), bins=50, color=BLUE, alpha=0.8, edgecolor='none')
ax2.axvline(delayed_only.mean(), color=CYAN, linestyle='--',
            label=f'Media: {delayed_only.mean():.0f} min')
ax2.axvline(delayed_only.median(), color=RED, linestyle='--',
            label=f'Mediana: {delayed_only.median():.0f} min')
ax2.set_xlabel('Minutos de retraso', color='white')
ax2.set_ylabel('Cantidad de vuelos', color='white')
ax2.set_title('Distribución de minutos de retraso', color='white')
ax2.legend(facecolor='#161b22', labelcolor='white')

plt.tight_layout()
plt.show()
print(f'\n📊 Resumen:')
print(f'   Vuelos retrasados >15 min: {delayed_pct:.1f}%')
print(f'   Retraso promedio (cuando hay retraso): {delayed_only.mean():.0f} min')
print(f'   80% de retrasos < {delayed_only.quantile(.8):.0f} min')

In [ ]:
airline_stats = (
    df_plot
    .groupby('AIRLINE')
    .agg(
        total_flights=('ARRIVAL_DELAY', 'count'),
        avg_arrival_delay=('ARRIVAL_DELAY', 'mean'),
        total_delay_minutes=('ARRIVAL_DELAY', lambda values: values.clip(lower=0).sum()),
    )
    .merge(airlines.rename(columns={'IATA_CODE': 'AIRLINE', 'AIRLINE': 'AIRLINE_NAME'}), on='AIRLINE', how='left')
    .sort_values('avg_arrival_delay', ascending=False)
)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle('Retrasos por Aerolínea', color='white', fontsize=14, fontweight='bold')

axes[0].barh(
    airline_stats['AIRLINE_NAME'].fillna(airline_stats['AIRLINE']),
    airline_stats['avg_arrival_delay'],
    color=BLUE,
)
axes[0].set_xlabel('Retraso promedio de llegada (min)', color='white')
axes[0].set_title('Retraso promedio por aerolínea', color='white')

airline_sorted2 = airline_stats.sort_values('total_delay_minutes', ascending=False)
axes[1].barh(
    airline_sorted2['AIRLINE_NAME'].fillna(airline_sorted2['AIRLINE']),
    airline_sorted2['total_delay_minutes'] / 1_000_000,
    color=CYAN,
    alpha=0.8,
)
axes[1].set_xlabel('Millones de minutos de retraso', color='white')
axes[1].set_title('Carga total de retraso por aerolínea', color='white')

plt.tight_layout()
plt.show()
print(airline_stats[['AIRLINE_NAME','total_flights','avg_arrival_delay','total_delay_minutes']].to_string(index=False))

In [ ]:
df_plot = df_plot.copy()
df_plot['HOUR'] = df_plot['SCHEDULED_DEPARTURE'] // 100

hourly = df_plot.groupby('HOUR')['ARRIVAL_DELAY'].mean()
monthly = df_plot.groupby('MONTH')['ARRIVAL_DELAY'].mean()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Patrones Temporales de Retraso Promedio', color='white', fontsize=14, fontweight='bold')

bar_colors = [RED if h >= 15 else BLUE for h in hourly.index]
axes[0].bar(hourly.index, hourly.values, color=bar_colors)
axes[0].set_xlabel('Hora del día', color='white')
axes[0].set_ylabel('Retraso promedio de llegada (min)', color='white')
axes[0].set_title('Retraso promedio por hora del día', color='white')
axes[0].axvspan(15, 20, alpha=0.15, color=RED, label='Zona operativa 3–8 PM')
axes[0].legend(facecolor='#161b22', labelcolor='white')

month_names = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
axes[1].plot(monthly.index, monthly.values, color=CYAN, linewidth=2.5, marker='o')
axes[1].fill_between(monthly.index, monthly.values, alpha=0.2, color=CYAN)
axes[1].set_xticks(range(1, 13))
axes[1].set_xticklabels(month_names, color='white')
axes[1].set_ylabel('Retraso promedio de llegada (min)', color='white')
axes[1].set_title('Retraso promedio por mes', color='white')

plt.tight_layout()
plt.show()
print(f'\n⏰ Hora con mayor retraso promedio: {hourly.idxmax()}:00h ({hourly.max():.1f} min)')
print(f'📅 Mes con mayor retraso promedio: {month_names[monthly.idxmax()-1]} ({monthly.max():.1f} min)')

In [ ]:
delay_causes = {
    'Air System (NAS)' : 'AIR_SYSTEM_DELAY',
    'Late Aircraft'    : 'LATE_AIRCRAFT_DELAY',
    'Airline Internal' : 'AIRLINE_DELAY',
    'Weather'          : 'WEATHER_DELAY',
    'Security'         : 'SECURITY_DELAY'
}

cause_totals = {name: df_raw[col].sum() for name, col in delay_causes.items()}
cause_df = pd.Series(cause_totals).sort_values(ascending=True)
total_cause_min = cause_df.sum()

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(cause_df.index, cause_df.values / 1e6, color=[BLUE, CYAN, GREEN, RED, GRAY])
ax.set_xlabel('Millones de minutos de retraso total', color='white')
ax.set_title('Causas de Retraso — Total de Minutos en 2015', color='white', fontsize=13, fontweight='bold')

for bar, (name, val) in zip(bars, zip(cause_df.index, cause_df.values)):
    pct = val / total_cause_min * 100
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
            f'{pct:.1f}%', va='center', color='white', fontweight='bold')

plt.tight_layout()
plt.show()

print('\n📊 Causas de retraso (% de minutos totales):')
for name, val in sorted(cause_totals.items(), key=lambda x: -x[1]):
    print(f'   {name:20s}: {val/total_cause_min*100:.1f}% ({val/1e6:.1f}M min)')

In [ ]:
airport_stats = (
    df_plot.groupby('ORIGIN_AIRPORT')
    .agg(flights=('ARRIVAL_DELAY','count'), avg_delay_min=('ARRIVAL_DELAY','mean'))
    .query('flights >= 1000')
    .sort_values('avg_delay_min', ascending=False)
    .head(15)
)

df_plot['ROUTE'] = df_plot['ORIGIN_AIRPORT'] + ' → ' + df_plot['DESTINATION_AIRPORT']
route_stats = (
    df_plot.groupby('ROUTE')
    .agg(flights=('ARRIVAL_DELAY','count'), avg_delay_min=('ARRIVAL_DELAY','mean'))
    .query('flights >= 500')
    .sort_values('avg_delay_min', ascending=False)
    .head(10)
)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle('Aeropuertos y Rutas con Mayor Retraso Promedio', color='white', fontsize=13, fontweight='bold')

axes[0].barh(airport_stats.index, airport_stats['avg_delay_min'], color=RED, alpha=0.85)
axes[0].set_xlabel('Retraso promedio de llegada (min)', color='white')
axes[0].set_title('Top 15 aeropuertos origen', color='white')

axes[1].barh(route_stats.index, route_stats['avg_delay_min'], color=CYAN, alpha=0.85)
axes[1].set_xlabel('Retraso promedio de llegada (min)', color='white')
axes[1].set_title('Top 10 rutas con mayor retraso promedio', color='white')

plt.tight_layout()
plt.show()

## ⚙️ 3. Limpieza y Feature Engineering

In [ ]:
df = df_raw.copy()
df = df[df['CANCELLED'] == 0].copy()
df = df.dropna(subset=['ARRIVAL_DELAY']).copy()

delay_cols = ['AIR_SYSTEM_DELAY','SECURITY_DELAY','AIRLINE_DELAY',
              'LATE_AIRCRAFT_DELAY','WEATHER_DELAY']
df[delay_cols] = df[delay_cols].fillna(0)
df['DEPARTURE_DELAY'] = df['DEPARTURE_DELAY'].fillna(0)

print(f'Registros después de limpieza: {len(df):,}')
print(f'Nulos restantes: {df.isnull().sum().sum()}')

In [ ]:
df['HOUR_OF_DAY'] = df['SCHEDULED_DEPARTURE'] // 100
df['IS_PEAK_HOUR'] = df['HOUR_OF_DAY'].between(15, 20).astype(int)
df['IS_WEEKEND'] = df['DAY_OF_WEEK'].isin([6, 7]).astype(int)
df['CASCADING_DELAY_FLAG'] = (df['LATE_AIRCRAFT_DELAY'] > 0).astype(int)
df['ROUTE_AVG_DELAY'] = df.groupby(['ORIGIN_AIRPORT', 'DESTINATION_AIRPORT'])['ARRIVAL_DELAY'].transform('mean')
df['AIRLINE_AVG_DELAY'] = df.groupby('AIRLINE')['ARRIVAL_DELAY'].transform('mean')
df['IS_HIGH_SEASON'] = df['MONTH'].isin([6, 7, 8, 12, 1]).astype(int)

le = LabelEncoder()
df['AIRLINE_ENC'] = le.fit_transform(df['AIRLINE'].astype(str))

new_features = [
    'ARRIVAL_DELAY',
    'HOUR_OF_DAY',
    'IS_PEAK_HOUR',
    'IS_WEEKEND',
    'CASCADING_DELAY_FLAG',
    'ROUTE_AVG_DELAY',
    'AIRLINE_AVG_DELAY',
    'IS_HIGH_SEASON',
    'AIRLINE_ENC',
]

print('✅ Variables preparadas para regresión:')
for feature in new_features:
    print(f'   {feature}: {df[feature].dtype} | ejemplo: {df[feature].iloc[0]}')

## 🤖 4. Entrenamiento de Modelos de ML

## Modelo, variables, features y etiqueta

En esta seccion dejamos explicito el contrato del modelo para la presentacion:

- **Modelo principal:** XGBoost Regressor, elegido como modelo no lineal y no clasificador.
- **Problema:** estimar los minutos de retraso de llegada de un vuelo.
- **Etiqueta:** `ARRIVAL_DELAY`, variable numerica que representa los minutos de retraso de llegada.
- **Features:** variables temporales, operativas e historicas agregadas que describen horario, temporada, aerolinea, ruta, distancia y senales de demora.
- **Uso de negocio:** convertir los minutos estimados de retraso en decisiones operativas, alertas y priorizacion de recursos.

Nota tecnica: algunas variables como `DEPARTURE_DELAY`, `AIR_SYSTEM_DELAY`, `WEATHER_DELAY` y `CASCADING_DELAY_FLAG` pueden no estar disponibles antes de la salida del vuelo. En esta entrega se usan para demostrar el flujo completo de EDA, regresion, empaquetado, API y ngrok. En un sistema productivo se separaria un modelo pre-salida usando solamente variables conocidas antes del vuelo.

In [ ]:
FEATURES = [
    'HOUR_OF_DAY', 'IS_PEAK_HOUR', 'IS_WEEKEND', 'IS_HIGH_SEASON',
    'CASCADING_DELAY_FLAG', 'ROUTE_AVG_DELAY', 'AIRLINE_AVG_DELAY',
    'AIRLINE_ENC', 'DISTANCE', 'DEPARTURE_DELAY', 'MONTH', 'DAY_OF_WEEK',
    'AIR_SYSTEM_DELAY', 'WEATHER_DELAY'
]
TARGET = 'ARRIVAL_DELAY'

model_df = df[FEATURES + [TARGET]].replace([np.inf, -np.inf], np.nan).dropna().copy()
X = model_df[FEATURES]
y = model_df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'Registros para modelado: {len(model_df):,}')
print(f'Train: {len(X_train):,} registros')
print(f'Test: {len(X_test):,} registros')
print(f'Target: {TARGET}')
print('Features:')
for feature in FEATURES:
    print(f'  - {feature}')

In [ ]:
import time

results = {}

models = {
    'Random Forest Regressor': RandomForestRegressor(n_estimators=120, random_state=42, n_jobs=-1),
    'XGBoost Regressor': XGBRegressor(
        n_estimators=250,
        max_depth=6,
        learning_rate=0.08,
        subsample=0.9,
        colsample_bytree=0.9,
        random_state=42,
        objective='reg:squarederror',
    ),
}

for name, model in models.items():
    print(f'\n🚀 Entrenando {name}...')
    t0 = time.time()
    model.fit(X_train, y_train)
    elapsed = time.time() - t0
    y_pred = model.predict(X_test)
    mae = mean_absolute_error(y_test, y_pred)
    rmse = mean_squared_error(y_test, y_pred) ** 0.5
    r2 = r2_score(y_test, y_pred)

    results[name] = {
        'model': model,
        'y_pred': y_pred,
        'mae': mae,
        'rmse': rmse,
        'r2': r2,
        'elapsed': elapsed,
    }

    print(f'   ✅ Listo en {elapsed:.1f}s | MAE: {mae:.2f} min | RMSE: {rmse:.2f} min | R2: {r2:.3f}')

In [ ]:
comparison = []
for name, result in results.items():
    comparison.append({
        'Modelo': name,
        'MAE_min': round(result['mae'], 2),
        'RMSE_min': round(result['rmse'], 2),
        'R2': round(result['r2'], 3),
        'Tiempo_s': round(result['elapsed'], 1),
    })

comp_df = pd.DataFrame(comparison).sort_values(['MAE_min', 'RMSE_min'])
best = comp_df.iloc[0]['Modelo']

print('\n📊 TABLA COMPARATIVA DE MODELOS DE REGRESIÓN')
print('=' * 72)
print(comp_df.to_string(index=False))
print('=' * 72)
print(f'\n🏆 Mejor modelo: {best}')

In [ ]:
best_model_name = best
best_pred = results[best_model_name]['y_pred']
residuals = y_test - best_pred

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Evaluación del Modelo de Regresión', color='white', fontsize=13, fontweight='bold')

axes[0].scatter(y_test, best_pred, alpha=0.25, color=BLUE, s=12)
min_value = min(y_test.min(), best_pred.min())
max_value = max(y_test.max(), best_pred.max())
axes[0].plot([min_value, max_value], [min_value, max_value], color=RED, linestyle='--', linewidth=2)
axes[0].set_xlabel('Retraso real de llegada (min)', color='white')
axes[0].set_ylabel('Retraso estimado de llegada (min)', color='white')
axes[0].set_title(f'Real vs estimado — {best_model_name}', color='white')

axes[1].hist(residuals.clip(lower=-120, upper=120), bins=50, color=CYAN, alpha=0.85)
axes[1].axvline(0, color=RED, linestyle='--', linewidth=2)
axes[1].set_xlabel('Error residual real - estimado (min)', color='white')
axes[1].set_ylabel('Cantidad de vuelos', color='white')
axes[1].set_title('Distribución de errores', color='white')

plt.tight_layout()
plt.show()

In [ ]:
best_model = results[best_model_name]['model']
importances = pd.Series(best_model.feature_importances_, index=FEATURES).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))
colors_imp = [RED if index >= len(importances) - 3 else BLUE for index in range(len(importances))]
ax.barh(importances.index, importances.values * 100, color=colors_imp)
ax.set_xlabel('Importancia (%)', color='white')
ax.set_title(f'Feature Importance — {best_model_name}', color='white', fontsize=13, fontweight='bold')

for index, (feature, value) in enumerate(importances.items()):
    ax.text(value * 100 + 0.2, index, f'{value * 100:.1f}%', va='center', color='white', fontsize=9)

plt.tight_layout()
plt.show()

print('\n🔑 Top 5 features más importantes:')
for feature, importance in importances.sort_values(ascending=False).head(5).items():
    print(f'   {feature:30s}: {importance * 100:.1f}%')

In [ ]:
error_abs = np.abs(residuals)
error_summary = pd.DataFrame({
    'Metrica': ['MAE', 'RMSE', 'Mediana error absoluto', 'P80 error absoluto', 'P95 error absoluto'],
    'Minutos': [
        results[best_model_name]['mae'],
        results[best_model_name]['rmse'],
        np.median(error_abs),
        np.quantile(error_abs, 0.80),
        np.quantile(error_abs, 0.95),
    ],
})

print('📊 Resumen de error del modelo de regresión')
display(error_summary.round(2))

fig, ax = plt.subplots(figsize=(10, 5))
ax.boxplot(error_abs.clip(upper=180), vert=False, patch_artist=True, boxprops=dict(facecolor=BLUE, color=CYAN))
ax.set_xlabel('Error absoluto en minutos', color='white')
ax.set_title('Distribución del error absoluto', color='white', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
prediction_review = pd.DataFrame({
    'actual_delay_min': y_test.values,
    'predicted_delay_min': best_pred,
})
prediction_review['absolute_error_min'] = (prediction_review['actual_delay_min'] - prediction_review['predicted_delay_min']).abs()
prediction_review['risk_level'] = pd.cut(
    prediction_review['predicted_delay_min'],
    bins=[-np.inf, 15, 45, np.inf],
    labels=['low', 'medium', 'high'],
)

risk_summary = (
    prediction_review.groupby('risk_level', observed=True)
    .agg(
        flights=('predicted_delay_min', 'count'),
        avg_predicted_delay=('predicted_delay_min', 'mean'),
        avg_actual_delay=('actual_delay_min', 'mean'),
        mae=('absolute_error_min', 'mean'),
    )
    .round(2)
)

print('📊 Interpretación operativa por nivel de riesgo')
display(risk_summary)

display(prediction_review.sort_values('predicted_delay_min', ascending=False).head(10).round(2))

In [ ]:
from pathlib import Path
import sys
import joblib

PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.flight_delay_features import build_model_bundle

MODEL_DIR = Path('models')
MODEL_DIR.mkdir(exist_ok=True)
MODEL_PATH = MODEL_DIR / 'flight_delay_model.joblib'

model_bundle = build_model_bundle(
    results[best_model_name]['model'],
    model_name=best_model_name,
    features=FEATURES,
)

joblib.dump(model_bundle, MODEL_PATH)

print('✅ Modelo de regresión empaquetado correctamente')
print(f'Ruta: {MODEL_PATH.resolve()}')
print(f"Modelo: {model_bundle['model_name']}")
print(f"Target: {model_bundle['target']}")
print('Features usadas por la API:')
for feature in model_bundle['features']:
    print(f'  - {feature}')

## 📋 5. Conclusiones

| Hallazgo | Detalle |
|---|---|
| **Modelo ganador** | Modelo de regresion con menor MAE y RMSE |
| **Etiqueta** | `ARRIVAL_DELAY`, minutos de retraso de llegada |
| **Tipo de problema** | Regresion supervisada, no clasificacion |
| **Feature importante esperada** | Variables operativas e historicas como demoras previas, ruta y aerolinea |
| **Salida del modelo** | Minutos estimados de retraso de llegada |
| **Uso de negocio** | Priorizar acciones cuando la prediccion supera 15 o 45 minutos |

### Próximos pasos
- Integrar datos de clima en tiempo real via OpenWeatherMap API
- Agregar datos de congestion ATC por aeropuerto
- Separar un modelo pre-salida con variables disponibles antes del vuelo
- Implementar reentrenamiento automatico cada 3 meses